# Stack Overflow Dataset Integration

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
old_data_path = Path("../../data/raw/stackoverflow_full.csv")
new_data_path = Path("../../data/raw/stackoverflow_2025_results.csv")

In [3]:
old_df = pd.read_csv(old_data_path)
new_df = pd.read_csv(new_data_path, low_memory=False)

In [4]:
old_df.shape, new_df.shape

((73462, 15), (49191, 172))

In [5]:
new_df["EdLevel"].value_counts(dropna=False)

EdLevel
Bachelor’s degree (B.A., B.S., B.Eng., etc.)                                          20278
Master’s degree (M.A., M.S., M.Eng., MBA, etc.)                                       12589
Some college/university study without earning a degree                                 6182
Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)     3631
Professional degree (JD, MD, Ph.D, Ed.D, etc.)                                         2624
Associate degree (A.A., A.S., etc.)                                                    1562
NaN                                                                                    1042
Other (please specify):                                                                 701
Primary/elementary school                                                               582
Name: count, dtype: int64

In [6]:
education_level_map = {
    "Bachelor’s degree (B.A., B.S., B.Eng., etc.)": "Undergraduate",
    "Associate degree (A.A., A.S., etc.)": "Undergraduate",
    "Master’s degree (M.A., M.S., M.Eng., MBA, etc.)": "Master",
    "Professional degree (JD, MD, Ph.D, Ed.D, etc.)": "PhD",
    "Some college/university study without earning a degree": "Other",
    "Other (please specify):": "Other",
    "Secondary school (e.g. American high school, German Realschule or Gymnasium, etc.)": "NoHigherEd",
    "Primary/elementary school": "NoHigherEd",
}

In [7]:
new_df["EdLevelMapped"] = new_df["EdLevel"].map(education_level_map)

In [8]:
new_df[["EdLevel", "EdLevelMapped"]].drop_duplicates()

,EdLevel,EdLevelMapped
0,"Master’s degree (M.A., M.S., M.Eng., MBA, etc.)",Master
1,"Associate degree (A.A., A.S., etc.)",Undergraduate
2,"Bachelor’s degree (B.A., B.S., B.Eng., etc.)",Undergraduate
6,Some college/university study without earning ...,Other
7,"Professional degree (JD, MD, Ph.D, Ed.D, etc.)",PhD
98,"Secondary school (e.g. American high school, G...",NoHigherEd
112,Other (please specify):,Other
179,Primary/elementary school,NoHigherEd
370,NaN,NaN


In [9]:
employment_map = {
    "Employed": 1,
    "Independent contractor, freelancer, or self-employed": 1,
    "Student": 0,
    "Not employed": 0,
    "Retired": 0,
}

In [10]:
new_df["EmploymentMapped"] = new_df["Employment"].map(employment_map)

In [11]:
new_df["EmploymentMapped"].value_counts(dropna=False)

EmploymentMapped
1.0    40458
0.0     7363
NaN     1370
Name: count, dtype: int64

In [12]:
new_df["YearsCodeMapped"] = new_df["YearsCode"].clip(lower=0, upper=50)
new_df["YearsCodeProMapped"] = new_df["WorkExp"].clip(lower=0, upper=50)

In [13]:
new_df[
    ["YearsCode", "YearsCodeMapped", "WorkExp", "YearsCodeProMapped"]
].describe()

,YearsCode,YearsCodeMapped,WorkExp,YearsCodeProMapped
count,43042.000000,43042.000000,42893.000000,42893.000000
mean,16.570861,16.487756,13.367403,13.304432
std,11.787610,11.419379,10.800117,10.466358
min,1.000000,1.000000,1.000000,1.000000
25%,8.000000,8.000000,5.000000,5.000000
50%,14.000000,14.000000,10.000000,10.000000
75%,24.000000,24.000000,20.000000,20.000000
max,100.000000,50.000000,100.000000,50.000000


In [14]:
new_df["PreviousSalaryMapped"] = new_df["ConvertedCompYearly"]

In [15]:
new_df["PreviousSalaryMapped"].describe(
    percentiles=[0.01, 0.25, 0.50, 0.75, 0.99]
)

count    2.394700e+04
mean     1.017615e+05
std      4.617569e+05
min      1.000000e+00
1%       6.500000e+01
25%      3.817100e+04
50%      7.532000e+04
75%      1.205960e+05
99%      4.408560e+05
max      5.000000e+07
Name: PreviousSalaryMapped, dtype: float64

In [17]:
technology_source_columns = [
    "LanguageHaveWorkedWith",
    "DatabaseHaveWorkedWith",
    "PlatformHaveWorkedWith",
    "WebframeHaveWorkedWith",
    "SOTagsHaveWorkedWith",
    "AIAgentOrchestration",
    "AIAgentKnowledge",
    "AIAgentObserveSecure",
]

In [18]:
missing_technology_columns = [
    column
    for column in technology_source_columns
    if column not in new_df.columns
]

missing_technology_columns

[]

In [19]:
def merge_technology_values(row):
    merged_technologies = []

    for column in technology_source_columns:
        value = row[column]

        if pd.isna(value):
            continue

        for technology in str(value).split(";"):
            technology = technology.strip()

            if (
                technology
                and technology not in merged_technologies
            ):
                merged_technologies.append(technology)

    if not merged_technologies:
        return pd.NA

    return ";".join(merged_technologies)

In [20]:
new_df["HaveWorkedWithMapped"] = new_df.apply(
    merge_technology_values,
    axis=1
)

In [21]:
new_df[
    technology_source_columns + ["HaveWorkedWithMapped"]
].head()

,LanguageHaveWorkedWith,DatabaseHaveWorkedWith,PlatformHaveWorkedWith,WebframeHaveWorkedWith,SOTagsHaveWorkedWith,AIAgentOrchestration,AIAgentKnowledge,AIAgentObserveSecure,HaveWorkedWithMapped
0,Bash/Shell (all shells);Dart;SQL,Cloud Firestore;PostgreSQL,Amazon Web Services (AWS);Cloudflare;Firebase;...,NaN,NaN,Vertex AI,NaN,NaN,Bash/Shell (all shells);Dart;SQL;Cloud Firesto...
1,Java,Dynamodb;MongoDB,Amazon Web Services (AWS);Datadog;Docker;Homeb...,Spring Boot,NaN,NaN,NaN,NaN,Java;Dynamodb;MongoDB;Amazon Web Services (AWS...
2,Dart;HTML/CSS;JavaScript;TypeScript,MongoDB;MySQL;PostgreSQL,Datadog;Firebase;npm;pnpm,Next.js;Node.js;React,Google Gemini,NaN,Redis,NaN,Dart;HTML/CSS;JavaScript;TypeScript;MongoDB;My...
3,Java;Kotlin;SQL,NaN,Amazon Web Services (AWS);Google Cloud,Spring Boot,Amazon Bedrock,NaN,NaN,NaN,Java;Kotlin;SQL;Amazon Web Services (AWS);Goog...
4,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,Elasticsearch;Microsoft SQL Server;MySQL;Oracl...,Amazon Web Services (AWS);APT;Docker;Make;Mave...,Angular;ASP.NET;ASP.NET Core;Flask;jQuery,Large Language Model;Google Gemini,NaN,NaN,NaN,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...


In [22]:
def count_technologies(value):
    if pd.isna(value):
        return 0

    technologies = [
        technology.strip()
        for technology in value.split(";")
        if technology.strip()
    ]

    return len(technologies)

In [23]:
new_df["ComputerSkillsMapped"] = new_df[
    "HaveWorkedWithMapped"
].apply(count_technologies)

In [24]:
new_df[
    ["HaveWorkedWithMapped", "ComputerSkillsMapped"]
].head(10)

,HaveWorkedWithMapped,ComputerSkillsMapped
0,Bash/Shell (all shells);Dart;SQL;Cloud Firesto...,11
1,Java;Dynamodb;MongoDB;Amazon Web Services (AWS...,11
2,Dart;HTML/CSS;JavaScript;TypeScript;MongoDB;My...,16
3,Java;Kotlin;SQL;Amazon Web Services (AWS);Goog...,7
4,C;C#;C++;Delphi;HTML/CSS;Java;JavaScript;Lua;P...,37
5,Java;Scala;Amazon Web Services (AWS);Google Cl...,6
6,JavaScript;TypeScript;Amazon Web Services (AWS...,12
7,Bash/Shell (all shells);HTML/CSS;JavaScript;Py...,19
8,Java;Python;Scala;Cassandra;Databricks SQL;Duc...,20
9,<NA>,0


In [25]:
technology_alias_map = {
    "React": "React.js",
    "Amazon Web Services (AWS)": "AWS",
    "Bash/Shell (all shells)": "Bash/Shell",
    "Dynamodb": "DynamoDB",
    "Lisp": "LISP",
    "Neo4J": "Neo4j",
}

In [26]:
def normalize_technology_names(value):
    if pd.isna(value):
        return pd.NA

    normalized_technologies = []

    for technology in value.split(";"):
        technology = technology.strip()

        normalized_name = technology_alias_map.get(
            technology,
            technology
        )

        if (
            normalized_name
            and normalized_name not in normalized_technologies
        ):
            normalized_technologies.append(normalized_name)

    if not normalized_technologies:
        return pd.NA

    return ";".join(normalized_technologies)

In [27]:
new_df["HaveWorkedWithMapped"] = new_df[
    "HaveWorkedWithMapped"
].apply(normalize_technology_names)

In [28]:
new_df["CountryMapped"] = new_df["Country"].str.strip()

In [29]:
new_aligned_df = pd.DataFrame({
    "Country": new_df["CountryMapped"],
    "EdLevel": new_df["EdLevelMapped"],
    "YearsCode": new_df["YearsCodeMapped"],
    "YearsCodePro": new_df["YearsCodeProMapped"],
    "PreviousSalary": new_df["PreviousSalaryMapped"],
    "HaveWorkedWith": new_df["HaveWorkedWithMapped"],
    "ComputerSkills": new_df["ComputerSkillsMapped"],
    "Employment": new_df["EmploymentMapped"],
    "DataSource": "stackoverflow_2025",
})

In [30]:
old_aligned_df = old_df[
    [
        "Country",
        "EdLevel",
        "YearsCode",
        "YearsCodePro",
        "PreviousSalary",
        "HaveWorkedWith",
        "ComputerSkills",
        "Employment",
    ]
].copy()

old_aligned_df["DataSource"] = "legacy_dataset"

In [31]:
old_aligned_df.columns.tolist() == new_aligned_df.columns.tolist()

True

In [32]:
combined_df = pd.concat(
    [
        old_aligned_df,
        new_aligned_df
    ],
    ignore_index=True
)

In [33]:
combined_df.shape

(122653, 9)

In [34]:
combined_df["DataSource"].value_counts()

DataSource
legacy_dataset        73462
stackoverflow_2025    49191
Name: count, dtype: int64

In [35]:
combined_df.head()

,Country,EdLevel,YearsCode,YearsCodePro,PreviousSalary,HaveWorkedWith,ComputerSkills,Employment,DataSource
0,Sweden,Master,7.0,4.0,51552.0,C++;Python;Git;PostgreSQL,4,1.0,legacy_dataset
1,Spain,Undergraduate,12.0,5.0,46482.0,Bash/Shell;HTML/CSS;JavaScript;Node.js;SQL;Typ...,12,1.0,legacy_dataset
2,Germany,Master,15.0,6.0,77290.0,C;C++;Java;Perl;Ruby;Git;Ruby on Rails,7,1.0,legacy_dataset
3,Canada,Undergraduate,9.0,6.0,46135.0,Bash/Shell;HTML/CSS;JavaScript;PHP;Ruby;SQL;Gi...,13,1.0,legacy_dataset
4,Singapore,PhD,40.0,30.0,160932.0,C++;Python,2,0.0,legacy_dataset


In [36]:
processed_data_path = Path(
    "../../data/processed/stackoverflow_combined.csv"
)

processed_data_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

combined_df.to_csv(
    processed_data_path,
    index=False
)

In [37]:
processed_data_path.exists()

True

In [38]:
combined_check_df = pd.read_csv(processed_data_path)

combined_check_df.shape

(122653, 9)

In [39]:
combined_quality_checks = {
    "duplicate_rows": int(combined_df.duplicated().sum()),
    "invalid_experience_rows": int(
        (
            combined_df["YearsCodePro"]
            > combined_df["YearsCode"]
        ).sum()
    ),
    "negative_salary_rows": int(
        (combined_df["PreviousSalary"] < 0).sum()
    ),
    "salary_below_1000": int(
        (combined_df["PreviousSalary"] < 1000).sum()
    ),
    "missing_country": int(
        combined_df["Country"].isna().sum()
    ),
    "missing_education": int(
        combined_df["EdLevel"].isna().sum()
    ),
    "missing_technologies": int(
        combined_df["HaveWorkedWith"].isna().sum()
    ),
}

combined_quality_checks

{'duplicate_rows': 11160,
 'invalid_experience_rows': 4136,
 'negative_salary_rows': 0,
 'salary_below_1000': 1204,
 'missing_country': 13754,
 'missing_education': 1042,
 'missing_technologies': 16785}

In [40]:
combined_df.groupby("DataSource").size()

DataSource
legacy_dataset        73462
stackoverflow_2025    49191
dtype: int64